## Notebook Phase 1: Data Ingestion and Sourcing Architecture

### Task 1.1: Environment Initialization and Configuration

In [ ]:
import gc
import os
import sys
import time
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import missingno as msno
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.graphics.correlation as sgc
import statsmodels.stats.api as sms
import torch
from langdetect import LangDetectException, detect
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sqlalchemy import create_engine, text
from statsmodels.graphics.gofplots import qqplot
from statsmodels.stats.outliers_influence import OLSInfluence
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

### Intro

This dataset is from the kaggle dataset Youtube Trending Videos Dataset which is updated daily, here is the link [https://www.kaggle.com/datasets/canerkonuk/youtube-trending-videos-global] . The dataset contains information about trending Youtube videos, including details about the video and their respective channels.
The first huddle i stumbled upon was with the way the data was scrapped, I had 1.1 million entries most of which was multiple entries of a single video, example: The latest Drakes albums definitely trended for multiple days in multiple countries and the scrapper recorded each instance of this, creating lot's of noise in our data, which i had to get rid of. Details on how i handled this and came up with the view vw_final_modelling_ready, which is a much cleaner version will be in the folder SQL_commands/phase_001

### Task 1.2: Connection to my Database on Postgres

In [ ]:
# Tells Python to look one folder up (the root folder)
sys.path.append(os.path.abspath(os.path.join("..")))


from db_connect import connect_to_db

# Connect to the database
engine = connect_to_db()
# Query database tables natively into Pandas
if engine:
    query = "SELECT * FROM youtube_data_schema.vw_final_modeling_ready;"

    with engine.connect() as conn:
        df = pd.read_sql_query(query, conn)

    print("Database connection successful! Previewing data:")
    display(df.head())

### Task 1.3: Checking for Missing Values

In [ ]:
msno.matrix(df)

## Notebook Phase 2: Advanced ETL, Schema Enforcement, and Multilingual Translation

### Phase Overview

After cleaning and organizing our original dataset of approximately **1.1 million records**, we reduced it to **96,671 high-quality records** that accurately capture how videos trend across different regions and over time.

In this phase, we move beyond data cleaning and focus on preparing the information for deeper analysis and future predictive models. To achieve this, we divide the dataset into several specialized views, each designed to answer a specific business or research question.

These views act as organized data layers that make it easier to explore trends, identify opportunities, and support future machine learning and language analysis projects.

---
## 2.2 Analytical View Layer & Data Mart Materialization
## A. Data Views and Their Purpose

#### I. Clean Data Foundation (`vw_base_clean`)

**Purpose:**  
This serves as the master cleaned dataset. It standardizes text, converts video durations into a consistent format, and ensures that all values are stored correctly.

**Why it matters:**  
Having a reliable foundation ensures that all future analysis is based on accurate and consistent information, reducing errors and improving confidence in results.

---

#### II. Market Opportunity Analysis (`vw_market_opportunity` and `vw_kenya_global_comparison`)

**Purpose:**  
These views summarize how different video categories perform across countries and regions. They measure factors such as popularity, engagement levels, and competition.

**Why it matters:**  
They help identify content categories with strong audience interest and reveal differences between Kenyan trends and global trends, making it easier to spot growth opportunities.

---

#### III. Viral Content and Trend Analysis (`vw_breakthrough_video` and `vw_content_decay`)

**Purpose:**  
These views focus on videos that significantly outperform expectations and examine how quickly content gains popularity.

**Why it matters:**  
Understanding what makes a video go viral and how long that popularity lasts can help creators, marketers, and researchers identify successful content strategies.

---

#### IV. Creator Performance Profiles (`vw_channel_features`)

**Purpose:**  
This view combines information about each creator or channel into a single profile, including audience size, engagement levels, and historical performance.

**Why it matters:**  
It allows us to compare creators, identify different types of channels, and discover patterns among successful content producers.

---

#### V. Text and Language Analysis (`vw_nlp_features` and `vw_translation_queue`)

**Purpose:**  
These views collect text-based information such as video titles, descriptions, and tags, while also identifying content that may need translation.

**Why it matters:**  
This prepares the dataset for language analysis, helping us understand how wording, topics, and communication styles influence video performance. It also enables multilingual analysis by translating content into a common language when necessary.

---

## Execution Strategy

In the next stage, we use SQL scripts to create these analytical views within our PostgreSQL database. Once the views are successfully built and validated, they become the primary data sources for exploratory analysis, visualizations, and future predictive modeling work.

**SQL Script:** `SQL_command/phase_2.0.sql`

### B. Materialize the vw_translation_queue to a Table.

At this stage, we'll turn **vw_translation_queue** into a concrete physical staging table **(tbl_translation_workspace)** to handle text modifications safely.

**SQL Script:** `SQL_command/phase_2.1.sql`

### Tasks: 2.3 & 2.4: Language Detection & AI Translation Engine

Imagine you are managing an international mailroom with thousands of letters arriving from all over the world. To process them efficiently, you could hire one person to walk through the entire room just sorting the letters into language piles, and then hire a second person to walk through all those piles a second time to translate them. 

Instead of making two slow, separate trips across the room, our system handles both tasks on the spot, one video at a time. This "smart loop" design saves immense computer memory and significantly speeds up our data pipeline.

---

## How the Smart Pipeline Works (Step-by-Step)

### 1. Language Detection (Task 2.3) — "What language is this?"
The system picks up a video title or description and reads it instantly. It acts like a digital linguist, analyzing the words to figure out what language is being used (for example, identifying Swahili, French, or Spanish). It then tags the video with its correct language label.

### 2. AI Translation & Slang Normalization (Task 2.4) — "Translate it to English"
Once the language is identified, the system passes the text directly to a powerful translation AI (Meta's advanced translation model). 
* **For Foreign Text:** The AI automatically translates the title and description into clean English.
* **For Local Dialects:** It uses smart fallback rules to accurately interpret localized Kenyan expressions and slang (like Sheng), ensuring the true meaning isn't lost in translation.
* **For Native English:** If a video is already in English, the system smartly skips the translation step entirely. This prevents the computer from wasting valuable processing power on work that is already done.

---

## Why This Matters for Our Project
By converting every trending video title and description into a single, standardized language (English), we create a perfectly consistent text sandbox. This uniform dataset allows our upcoming AI tracking models to accurately analyze global content trends, emotional tones, and viewer behaviors without getting tripped up by language barriers.

In [ ]:
# ============================================================================
# 1. INITIALIZE DATABASE CONNECTION
# ============================================================================
db_url = (
    f"postgresql://{quote_plus(os.getenv('DB_USER'))}:"
    f"{quote_plus(os.getenv('DB_PASSWORD'))}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(db_url)

# ============================================================================
# 2. LOAD NLLB MODEL DIRECTLY (pipeline API doesn't support dynamic src_lang)
# ============================================================================
print("Loading Meta NLLB-200 Transformer Model...")
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()
print(f"Model loaded on [{device.upper()}]")

# ============================================================================
# 3. COMPREHENSIVE ISO -> NLLB LANGUAGE CODE MAP
# ============================================================================
NLLB_LANG_CODES = {
    "af": "afr_Latn",
    "ak": "aka_Latn",
    "am": "amh_Ethi",
    "ar": "arb_Arab",
    "az": "azj_Latn",
    "be": "bel_Cyrl",
    "bg": "bul_Cyrl",
    "bn": "ben_Beng",
    "bs": "bos_Latn",
    "ca": "cat_Latn",
    "cs": "ces_Latn",
    "cy": "cym_Latn",
    "da": "dan_Latn",
    "de": "deu_Latn",
    "el": "ell_Grek",
    "en": "eng_Latn",
    "es": "spa_Latn",
    "et": "est_Latn",
    "fa": "pes_Arab",
    "fi": "fin_Latn",
    "fr": "fra_Latn",
    "ga": "gle_Latn",
    "gl": "glg_Latn",
    "gu": "guj_Gujr",
    "he": "heb_Hebr",
    "hi": "hin_Deva",
    "hr": "hrv_Latn",
    "hu": "hun_Latn",
    "hy": "hye_Armn",
    "id": "ind_Latn",
    "is": "isl_Latn",
    "it": "ita_Latn",
    "ja": "jpn_Jpan",
    "ka": "kat_Geor",
    "kk": "kaz_Cyrl",
    "km": "khm_Khmr",
    "kn": "kan_Knda",
    "ko": "kor_Hang",
    "lt": "lit_Latn",
    "lv": "lvs_Latn",
    "mk": "mkd_Cyrl",
    "ml": "mal_Mlym",
    "mn": "khk_Cyrl",
    "mr": "mar_Deva",
    "ms": "zsm_Latn",
    "mt": "mlt_Latn",
    "my": "mya_Mymr",
    "ne": "npi_Deva",
    "nl": "nld_Latn",
    "no": "nob_Latn",
    "pa": "pan_Guru",
    "pl": "pol_Latn",
    "pt": "por_Latn",
    "ro": "ron_Latn",
    "ru": "rus_Cyrl",
    "si": "sin_Sinh",
    "sk": "slk_Latn",
    "sl": "slv_Latn",
    "so": "som_Latn",
    "sq": "als_Latn",
    "sr": "srp_Cyrl",
    "sv": "swe_Latn",
    "sw": "swh_Latn",
    "ta": "tam_Taml",
    "te": "tel_Telu",
    "th": "tha_Thai",
    "tl": "tgl_Latn",
    "tr": "tur_Latn",
    "uk": "ukr_Cyrl",
    "ur": "urd_Arab",
    "uz": "uzn_Latn",
    "vi": "vie_Latn",
    "zh": "zho_Hans",
    "zh-cn": "zho_Hans",
    "zh-tw": "zho_Hant",
}

# ============================================================================
# 4. CORE BATCH TRANSLATION FUNCTION (true batching, grouped by language)
# ============================================================================


def translate_batch(texts, src_nllb_code, max_length=256, batch_size=16):
    """Translate a list of texts from src_nllb_code to English in batches."""
    results = list(texts)

    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]

        tokenizer.src_lang = src_nllb_code
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)

        target_lang_id = tokenizer.convert_tokens_to_ids("eng_Latn")

        with torch.no_grad():
            generated = model.generate(
                **encoded,
                forced_bos_token_id=target_lang_id,
                max_length=max_length,
                num_beams=1,  # greedy decoding — 5x less VRAM than default beam search
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        results[start : start + len(batch)] = decoded

        del encoded, generated
        if device == "cuda":
            torch.cuda.empty_cache()

    return results


# ============================================================================
# 5. BATCH PROCESSING LOOP (fetch → group by language → batch translate → upsert)
# ============================================================================


def process_translation_batch(batch_size=200):
    query = f"""
        SELECT video_id, video_trending_country, video_title, video_description
        FROM youtube_data_schema.tbl_translation_workspace
        WHERE translation_complete IS NOT TRUE
           OR translation_complete IS NULL
        LIMIT {batch_size};
    """

    with engine.connect() as conn:
        df = pd.read_sql_query(text(query), conn)

    if df.empty:
        return False  # Queue exhausted

    print(f"\nProcessing batch of {len(df)} records...")

    # --- Detect language per row ---
    def detect_iso(text):
        try:
            code = detect(str(text))
            # Normalise Chinese variants
            if code.startswith("zh"):
                return "zh"
            return code
        except LangDetectException:
            return "en"  # fallback to English (skip translation)

    df["detected_iso"] = df["video_title"].apply(detect_iso)
    df["nllb_src"] = df["detected_iso"].map(NLLB_LANG_CODES).fillna("eng_Latn")

    # Initialise output columns with original values (safe fallback)
    df["translated_title"] = df["video_title"]
    df["translated_desc"] = df["video_description"].fillna("")

    # --- Group by language and batch translate (avoids reinitialising tokenizer per row) ---
    non_english = df[df["detected_iso"] != "en"]
    for nllb_code, group in tqdm(non_english.groupby("nllb_src"), desc="Languages"):
        idxs = group.index.tolist()

        # Translate titles
        raw_titles = group["video_title"].tolist()
        translated_titles = translate_batch(raw_titles, nllb_code, max_length=128)
        df.loc[idxs, "translated_title"] = translated_titles

        # Translate descriptions (truncate to 500 chars to control sequence length)
        raw_descs = [
            str(d)[:500] if pd.notna(d) and str(d).strip() else ""
            for d in group["video_description"]
        ]
        non_empty_mask = [bool(d.strip()) for d in raw_descs]

        if any(non_empty_mask):
            descs_to_translate = [d for d, m in zip(raw_descs, non_empty_mask) if m]
            translated_descs = translate_batch(descs_to_translate, nllb_code, max_length=256)

            translated_iter = iter(translated_descs)
            df.loc[idxs, "translated_desc"] = [
                next(translated_iter) if m else d for d, m in zip(raw_descs, non_empty_mask)
            ]

        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    # --- Bulk upsert back to DB (one transaction per batch, not per row) ---
    update_query = text(
        """
        UPDATE youtube_data_schema.tbl_translation_workspace
        SET
            detected_language    = :detected_lang,
            translated_title     = :trans_title,
            translated_description = :trans_desc,
            translation_complete = TRUE
        WHERE video_id = :v_id
          AND video_trending_country = :country;
    """
    )

    with engine.begin() as conn:
        conn.execute(
            update_query,
            [
                {
                    "detected_lang": row.detected_iso,
                    "trans_title": row.translated_title,
                    "trans_desc": row.translated_desc,
                    "v_id": row.video_id,
                    "country": row.video_trending_country,
                }
                for row in df.itertuples()
            ],
        )

    print(f"✅ Batch committed to DB — {len(df)} rows updated.")
    return True


# ============================================================================
# 6. PIPELINE EXECUTION LOOP
# ============================================================================
batch_count = 1

while True:
    start_time = time.time()
    has_more = process_translation_batch(batch_size=200)

    if not has_more:
        print("\nTranslation pipeline complete! All rows processed.")
        break

    elapsed = round(time.time() - start_time, 2)
    print(f"Batch {batch_count} completed in {elapsed}s")
    batch_count += 1

### Task 2.5: Production-Ready Data Harmonization and Export

### Check For Missing Data

In [ ]:
from db_connect import connect_to_db

if engine:
    query = "SELECT * FROM youtube_data_schema.tbl_translation_workspace;"
    with engine.connect() as conn:
        translation_df = pd.read_sql_query(query, conn)

    print("Connection Succesful!")
    display(translation_df)

In [ ]:
# Convert all empty spaces or blank text blocks into true NaN/NULL values
translation_df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

# Re-run your matrix plot
msno.matrix(translation_df)

In [ ]:
missing_description = translation_df["video_description"].isnull().sum()
missing_description

So I'll fill all missing entries with **No Description Available**. This keeps the row intact for modeling other features (like view counts or upload hours) without causing string processing errors:

In [ ]:
# Replace description NaN values with a standard placeholder string
translation_df["video_description"] = translation_df["video_description"].fillna(
    "[No Description Available]"
)

translation_df["translated_description"] = translation_df["translated_description"].fillna(
    "[No Description Available]"
)

In [ ]:
msno.matrix(translation_df)

Replace this version of the table on postgres with this updated version.

In [ ]:
translation_df.to_sql(
    name="tbl_translation_workspace",
    con=engine,
    schema="youtube_data_schema",
    if_exists="replace",
    index=False,
)
print("Postgres table successfully overwritten")

# Phase 3: Feature Engineering

In this phase, we transform our raw, cleaned YouTube data into **features** —
numeric or categorical signals that a machine learning model can actually
learn from. Raw text and raw counts aren't directly useful to a model; we
need to translate them into measurable patterns.

We'll build four groups of features:
1. **Title structure** — how a title is "shaped" (length, caps, punctuation)
2. **Title sentiment** — how positive/negative/neutral a title sounds
3. **Engagement ratios** — likes and comments relative to views
4. **Virality signals** — whether a video broke far beyond its channel's normal reach

Along the way, we'll also build in a few **honesty checks** — features and
diagnostics whose only job is to tell us when our other features might be
unreliable. This matters because some of our data went through machine
translation, and translated text doesn't always behave the same way as
original text when we run analysis on it.

In [ ]:
nltk.download("vader_lexicon", quiet=True)

## Step 2: Confirm Available Columns

Good news — checking the database columns reveals two important things:

1. **`video_comment_count` exists**, so we can build a comment density
   feature for real (Step 9 below).
2. **The database already has precomputed ratio columns**:
   `like_view_ratio`, `comment_density`, and `subscriber_breakthrough_ratio`.

That second point changes our plan. Rather than recomputing these ratios
ourselves in pandas — which risks producing a *slightly different* number
than the database version (due to rounding, edge-case handling, or a
different formula entirely) — we'll **trust and reuse the database's
version**. This is a good general habit: if a value already exists
upstream, recomputing it downstream usually just creates two sources of
truth that can quietly drift apart and confuse anyone debugging the
pipeline later.

We'll still rename these columns to match the naming style we've been
using (`calculated_like_ratio`, etc.) so the rest of the notebook reads
consistently, but the actual numbers will come straight from SQL.

In [ ]:
column_check_query = """
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'youtube_data_schema'
      AND table_name = 'vw_base_clean'
    ORDER BY ordinal_position;
"""
with engine.connect() as conn:
    df_columns = pd.read_sql_query(text(column_check_query), conn)

print(df_columns.to_string())

## Step 3: Fetch the Feature Store

This query joins our cleaned base data (`vw_base_clean`) with our
translation results (`tbl_translation_workspace`). We use a `LEFT JOIN`
rather than a regular `JOIN` because we want to **keep every video**, even
ones that don't have a translation row yet — we'll handle those missing
values explicitly in the next step instead of silently losing rows.

Since `vw_base_clean` already includes `like_view_ratio`, `comment_density`,
and `subscriber_breakthrough_ratio`, those columns come along automatically
with `b.*` — no extra joins needed.

In [ ]:
from sqlalchemy import text

if engine:  # Check the engine instead of the volatile connection object
    fetch_query = """
        SELECT b.*, t.translated_title, t.translated_description
        FROM youtube_data_schema.vw_base_clean b
        LEFT JOIN youtube_data_schema.tbl_translation_workspace t
          ON b.video_id = t.video_id AND b.video_trending_country = t.video_trending_country;
    """

    print("📥 Fetching baseline feature store from PostgreSQL...")

    # Establish a fresh connection context that closes itself automatically
    with engine.connect() as conn:
        df_features = pd.read_sql_query(text(fetch_query), conn)

    print(f"✅ Loaded {len(df_features)} rows.")
    print(df_features.head())

## Step 4: Handle Missing Translations

Some rows won't have a match in the translation table (maybe translation
hasn't run for them yet, or they were already in English). Rather than
leaving these as empty/`NaN` values — which would break later string
operations — we fill them in with sensible defaults:

- Missing description → a placeholder text
- Missing translated title → fall back to the original title

In [ ]:
df_features["translated_description"] = df_features["translated_description"].fillna(
    "No Description Available"
)
df_features["translated_title"] = df_features["translated_title"].fillna(df_features["video_title"])

print("✅ Missing values handled.")

## Step 5: Flag Which Rows Were Actually Translated

Some titles in our dataset were genuinely translated by our model, and
some just got the original title copied over (no translation existed, or
it was already English). These two groups aren't the same for analysis
purposes — machine translation can flatten out tone, slang, or emotional
emphasis.

We record this as its own feature: `is_translated`. This lets us check
later whether other features (like sentiment) behave differently across
the two groups, rather than quietly assuming they're equally trustworthy.

In [ ]:
df_features["is_translated"] = (
    df_features["translated_title"] != df_features["video_title"]
).astype(int)

translated_share = df_features["is_translated"].mean() * 100
print(f"{translated_share:.1f}% of rows are machine-translated titles.")

## Step 6: Title Structure Features

Now we measure the "shape" of each title — creators trying to maximize
clicks often format titles in recognizable patterns:

- **Character/word count** — short and punchy vs. long and descriptive
- **Caps ratio** — what fraction of the title is in ALL CAPS (a classic
  attention-grabbing signal)
- **Brackets** — does it use `[MUST WATCH]` or `(2024)` style framing
- **Exclamation marks** — and specifically, *how many*, not just whether
  there's at least one. A title with `!!!` reads as more intense than one
  with a single `!`, so counting instead of just flagging keeps that extra
  information instead of throwing it away.